In [0]:
dbutils.widgets.text("fecha_corte", "2026-11-05")
dbutils.widgets.dropdown("modo", "incremental", ["incremental", "full", "reproceso"])
dbutils.widgets.dropdown("env", "dev", ["dev", "qa", "prod"])
dbutils.widgets.text("run_id", "")

In [0]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.common.config import PipelineConfig
from src.common.control import ejecucion
from src.gold.proceso import construir_y_publicar

params = dict(
    fecha_corte=dbutils.widgets.get("fecha_corte"),
    modo=dbutils.widgets.get("modo"),
    env=dbutils.widgets.get("env"),
)
if dbutils.widgets.get("run_id"):
    params["run_id"] = dbutils.widgets.get("run_id")

cfg = PipelineConfig(**params)

with ejecucion(spark, cfg, "gold"):
    resultado = construir_y_publicar(spark, cfg)

display(spark.createDataFrame(resultado))

In [0]:
%sql
SELECT estado_al_corte, COUNT(*) AS polizas, SUM(prima_emitida) AS prima_emitida
FROM workspace.default.gold_poliza_360
WHERE fecha_corte = (SELECT max(fecha_corte) FROM workspace.default.gold_poliza_360)
GROUP BY estado_al_corte ORDER BY polizas DESC;

In [0]:
%sql
SELECT date_format(fecha_inicio_vigencia, 'yyyy-MM') AS mes_emision, producto, sucursal,
       SUM(prima_emitida) AS emitida, SUM(prima_recaudada) AS recaudada, SUM(prima_pendiente) AS pendiente
FROM workspace.default.gold_poliza_360
WHERE fecha_corte = (SELECT max(fecha_corte) FROM workspace.default.gold_poliza_360)
GROUP BY ALL ORDER BY mes_emision DESC, producto, sucursal;

In [0]:
%sql
SELECT periodo,
       SUM(num_siniestros)   AS siniestros,
       SUM(costo_incurrido)  AS costo_incurrido,
       SUM(prima_devengada)  AS prima_devengada,
       ROUND(SUM(costo_incurrido) / NULLIF(SUM(prima_devengada), 0), 4) AS razon_siniestralidad
FROM workspace.default.gold_siniestralidad_mensual
WHERE fecha_corte = (SELECT max(fecha_corte) FROM workspace.default.gold_siniestralidad_mensual)
GROUP BY periodo ORDER BY periodo DESC;